# Proyecto Final: Predicción de Fuga de Clientes

El operador de telecomunicaciones **Interconnect** busca reducir la cancelación de clientes (`Churn`) mediante un sistema que identifique con anticipación a quienes podrían darse de baja, para ofrecerles promociones y planes especiales.

**Objetivo:** Desarrollar y comparar modelos capaces de predecir si un cliente se dará de baja próximamente (Sí/No).

- El modelo se evaluará principalmente con AUC-ROC y con Recall como métrica adicional.

## Etapa 1: Plan de Trabajo
* Antes de tratar de hac3er algo con los datos prompongo segmentación de clientes con KMeans y luego entrenar diferentes modelos de regresión logística y de clasificación
* Probar con
    * Logistic Regression
    * LinearSVC
    * LightBGM


*Escribe aquí tu plan de trabajo inicial. Aborda brevemente:*
1. *¿Cómo planeas unir los datos y qué harás con los valores nulos generados?*
2. *¿Cuál será tu variable objetivo y qué tipo de problema de Machine Learning resolverás?*
3. *¿Qué pasos de preprocesamiento (codificación categórica, fechas. etc.) consideras necesarios y sobre que variables?*
4. *¿Qué modelos planeas entrenar?*

## Etapa 2: Código de Solución

### 1. Exploración de Datos (EDA)

Descripción de los Datos

Los datos están divididos en cuatro archivos:

* `/datasets/final_provider/contract.csv`: Información del contrato (tipo de facturación. método de pago, fechas de inicio y fin).
* `/datasets/final_provider/personal.csv`: Datos demográficos del cliente.
* /datasets/final_provider/internet.csv: Información sobre los servicios de Internet contratados (fibra óptica, DSL. antivirus. etc.).
* `/datasets/final_provider/phone.csv`: Información sobre los servicios telefónicos (líneas múltiples).

*Carga de datos. análisis de distribuciones, identificación de anomalías.*

### 2. Preprocesamiento e Ingeniería de Características
*Procesar valores nulos, creación de la variable objetivo, codificación de variables categóricas (justifica tu elección de método).*
*Pista: Los modelos predictivos no entienden de fechas en formato texto. ¿Cómo puedes transformar las fechas de inicio y fin en una variable numérica útil para el modelo?*

### 3. Selección de Variables y Entrenamiento de Modelos (Baseline)
*Entrena al menos dos modelos distintos sin aplicar técnicas de balanceo de clases. Evalúa su AUC-ROC y Recall.*

### 4. Optimización y Manejo de Desbalance
*Aplica al menos una técnica para manejar el desbalance de clases (upsampling, downsampling, o ajuste de pesos) y busca los mejores hiperparámetros. Evalúa nuevamente.*

## Etapa 3: Informe de Solución
*Escribe aquí tu informe final para el equipo de negocio. Asegúrate de responder:*
1. *¿Qué modelo elegiste finalmente y por qué?*
2. *¿Cuáles fueron las métricas finales (AUC-ROC y Recall) en el conjunto de prueba?*
3. *En términos de negocio: ¿Qué significa tu valor de Recall? ¿Cómo impactaría tu modelo en la retención de clientes si el equipo de marketing lo utiliza hoy?*

# Carga y eploración

In [14]:
import warnings

import sys
import os
# Le dice python que busque liberrías ahí también
sys.path.append(os.path.join('src'))
import funciones_personales as fp

import pandas as pd
import numpy as np
import re

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    roc_auc_score
)

In [15]:
df_contract= pd.read_csv('datasets/final_provider/contract.csv')
df_internet= pd.read_csv('datasets/final_provider/internet.csv')
df_personal= pd.read_csv('datasets/final_provider/personal.csv')
df_phone= pd.read_csv('datasets/final_provider/phone.csv')

In [16]:
lista_dfs= [df_contract, df_internet, df_personal, df_phone]

# Cambia los nombres de columnas de todos los dataframes a snake_case
for i in lista_dfs:
    i.columns = [fp.to_snake_case((col)) for col in i.columns]

In [17]:
# imprime la información general de los df
for i in lista_dfs:
    print(df_personal.info())
    print(df_personal.head(5))
    print(df_personal.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     7043 non-null   object
 1   gender          7043 non-null   object
 2   senior_citizen  7043 non-null   int64 
 3   partner         7043 non-null   object
 4   dependents      7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB
None
  customer_id  gender  senior_citizen partner dependents
0  7590-VHVEG  Female               0     Yes         No
1  5575-GNVDE    Male               0      No         No
2  3668-QPYBK    Male               0      No         No
3  7795-CFOCW    Male               0      No         No
4  9237-HQITU  Female               0      No         No
customer_id       7043
gender               2
senior_citizen       2
partner              2
dependents           2
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entr

**df_contract**

* 7,043 datos con diferentes id sin valores nulos

1. Se crea la variable objetivo 'churn' = 1 si el cliente abandonó, churn=0 si sigue siendo cliente usando la columna `'end_date'`.
* Se aprecia que existe desbalance entre clases: 73% = 0, 27%  = 1. Se tratará después de la concatenación

2. Se transforman las fechas a datetime dejando el string "No" como NaT.
* OJO estas columnas contienen la respuesta directa por lo que hay que tener cuidado al implementar el modelo.

3. Se transforma `'total_charges` a float y se sustituyen los NaN por 0.0
* Solo son 11 clientes con un contrato muy reciente los cuales no ha llegado la fecha de facturación aún

**Resto de los df**

* Se comprueba que los customer_id existan en `'df_contracts`´
* Se define una lista de columnas con los datos de "Yes" y "No" transformados a 1 y 0 respectivamente ('gender' no se incluye)

In [18]:
# Se crea la variable objetivo
df_contract['churn']= (df_contract['end_date']!='No').astype(int)

# Transforma columnas begin_date y end_date a datetime
df_contract['begin_date']= pd.to_datetime(
    df_contract['begin_date'],
    format='%Y-%d-%m'
)
df_contract['end_date'] = pd.to_datetime(
    df_contract['end_date'].replace('No', pd.NaT),
    format='%Y-%d-%m %H:%M:%S'
)

# Transforma la columna total changes a float y rellena nan con 0
df_contract['total_charges']= pd.to_numeric(df_contract['total_charges'], errors='coerce').fillna(0.0)

In [19]:
# Revisa balance de clases
print(df_contract['churn'].value_counts())
df_contract['churn'].value_counts(normalize=True) * 100

churn
0    5174
1    1869
Name: count, dtype: int64


churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64

In [20]:
# muestra los clientes con total_charges = 0
fil= df_contract[df_contract['total_charges']==0]
print(fil.info())
print(fil.nunique())
print(fil['begin_date'].unique())
print(fil['churn'].unique())

<class 'pandas.core.frame.DataFrame'>
Index: 11 entries, 488 to 6754
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        11 non-null     object        
 1   begin_date         11 non-null     datetime64[ns]
 2   end_date           0 non-null      datetime64[ns]
 3   type               11 non-null     object        
 4   paperless_billing  11 non-null     object        
 5   payment_method     11 non-null     object        
 6   monthly_charges    11 non-null     float64       
 7   total_charges      11 non-null     float64       
 8   churn              11 non-null     int64         
dtypes: datetime64[ns](2), float64(2), int64(1), object(4)
memory usage: 880.0+ bytes
None
customer_id          11
begin_date            1
end_date              0
type                  2
paperless_billing     2
payment_method        3
monthly_charges      11
total_charges         1
churn      

In [21]:
# Compara los 'customer_id' de todos los df contra df_contracts y muestra si existen faltantes
for nombre, df in {
    'internet': df_internet,
    'personal': df_personal,
    'phone': df_phone
}.items():

    faltantes = df.loc[
        ~df['customer_id'].isin(df_contract['customer_id']),
        'customer_id'
    ]

    print(nombre, len(faltantes))

internet 0
personal 0
phone 0


In [22]:
# Lista de columnas binarias
binary_cols = [
    'paperless_billing',
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies',
    'partner',
    'dependents',
    'multiple_lines'
]

## 
for df in lista_dfs:
    cols = [col for col in binary_cols if col in df.columns]
    for col in cols:
        df[col] = df[col].map({'Yes': 1, 'No': 0})

In [23]:
print(df_contract.describe())


                          begin_date                       end_date  \
count                           7043                           1869   
mean   2016-11-22 16:33:40.048274688  2019-04-08 09:03:10.690208768   
min              2013-01-10 00:00:00            2019-01-10 00:00:00   
25%              2015-01-06 00:00:00            2019-01-11 00:00:00   
50%              2017-01-09 00:00:00            2019-01-11 00:00:00   
75%              2019-01-04 00:00:00            2019-01-12 00:00:00   
max              2020-01-02 00:00:00            2020-01-01 00:00:00   
std                              NaN                            NaN   

       paperless_billing  monthly_charges  total_charges        churn  
count        7043.000000      7043.000000    7043.000000  7043.000000  
mean            0.592219        64.761692    2279.734304     0.265370  
min             0.000000        18.250000       0.000000     0.000000  
25%             0.000000        35.500000     398.550000     0.000000  


## Resumen estadístico

El conjunto de datos `df_contract` contiene **7,043 clientes** e información sobre sus contratos, facturación, fechas de inicio y fin, cargos mensuales y cargos acumulados.

### Fechas de inicio y fin de los clientes

La variable `begin_date` contiene una fecha para los 7.043 clientes, abarcando el periodo de **2013 a 2020**. La mediana de la fecha de inicio se sitúa alrededor de enero de 2017, lo que indica que el conjunto de datos incluye clientes con duraciones de contrato muy variadas.

La variable `end_date` contiene fechas para **1,869 clientes**, mientras que los clientes restantes no tienen una fecha de fin registrada porque seguían activos en el momento que refleja el conjunto de datos.

El número de clientes con una `end_date` es coherente con la variable objetivo `churn` (abandono), donde:

* **1,869 clientes (26.54 %)** tienen `churn = 1`.
* **5,174 clientes (73.46 %)** tienen `churn = 0`.

Por lo tanto, la variable objetivo presenta un desequilibrio de clases moderado, aspecto que se abordará más adelante durante la fase de desarrollo del modelo.

### Cargos mensuales

`monthly_charges` es una variable numérica que oscila entre **18.25 y 118.75**.

| Estadístico        | Valor |
| ------------------ | ----: |
| Media              | 64.76 |
| Mediana            | 70.35 |
| Q1                 | 35.50 |
| Q3                 | 89.85 |
| Desviación estándar| 30.09 |

La diferencia entre la media y la mediana sugiere que la distribución no es perfectamente simétrica. No obstante, la variable tiene un rango numérico bien definido y se conservará como predictor continuo.

### Cargos totales

`total_charges` oscila entre **0 y 8,684.80**.

| Estadístico        | Valor |
| ------------------ | -------: |
| Media              | 2,279.73 |
| Mediana            | 1,394.55 |
| Q1                 | 398.55 |
| Q3                 | 3,786.60 |
| Desviación estándar | 2,266.79 |

La media es considerablemente superior a la mediana, lo que indica una distribución con sesgo a la derecha. Esto concuerda con el hecho de que los clientes tienen diferentes tiempos de permanencia: aquellos suscritos durante períodos más largos pueden acumular cargos totales sustancialmente mayores.

El conjunto de datos original contenía cadenas vacías en `total_charges`. Estos registros corresponden a clientes recién registrados que aún no habían recibido su primera factura y, por tanto, no habían realizado ningún pago. Estas observaciones no deben interpretarse automáticamente como datos erróneos o como un gasto nulo.

Por consiguiente, se conservará `total_charges` como un predictor potencial. Su relación con la antigüedad del cliente y la tasa de abandono (*churn*) deberá examinarse antes del entrenamiento del modelo.

### Facturación sin papel

`paperless_billing` es una variable binaria codificada como `0/1`.

Su media es **0.5922**, lo que indica que aproximadamente el **59.2 % de los clientes utiliza la facturación sin papel**.

### Variable objetivo

La variable objetivo `churn` es binaria:

* `0`: el cliente no abandonó el servicio.
* `1`: el cliente abandonó el servicio.

La distribución de la variable objetivo es:

| Abandono | Clientes | Porcentaje |
| -------- | -------: | ---------: |
| 0        | 5,174 | 73.46% |
| 1        | 1,869 | 26.54% |

Esta distribución de clases se tendrá en cuenta durante la fase de modelado. El desequilibrio de clases se abordará **una vez preparado el conjunto completo de características y antes del entrenamiento del modelo**, lo que permitirá evaluar los distintos modelos en condiciones comparables.

### Principales hallazgos

El conjunto de datos sobre contratos proporciona varios predictores potencialmente útiles para el abandono, en particular:

* cargos mensuales;
* cargos totales;
* tipo de contrato;
* método de pago;
* facturación sin papel;
* antigüedad del cliente, que puede derivarse de la información de fechas disponible.

No se utilizará `end_date` directamente como predictor, ya que contiene información sobre la salida del cliente y se empleó para definir la propia variable objetivo. Asimismo, `customer_id` se conservará únicamente como identificador para la integración de datos y no se utilizará como característica del modelo. La siguiente etapa consistirá en integrar los conjuntos de datos relativos a contratos, información personal, internet y telefonía; validar los identificadores de los clientes; crear variables derivadas adecuadas, como la antigüedad; codificar las variables categóricas; y preparar la matriz de características final antes de abordar el desequilibrio de clases y comparar los modelos de aprendizaje automático.

In [24]:
# Validación de fechas
print(df_contract[['begin_date', 'end_date']].dtypes)

print('\nBegin date:')
print(df_contract['begin_date'].describe())

print('\nEnd date:')
print(df_contract['end_date'].describe())

begin_date    datetime64[ns]
end_date      datetime64[ns]
dtype: object

Begin date:
count                             7043
mean     2016-11-22 16:33:40.048274688
min                2013-01-10 00:00:00
25%                2015-01-06 00:00:00
50%                2017-01-09 00:00:00
75%                2019-01-04 00:00:00
max                2020-01-02 00:00:00
Name: begin_date, dtype: object

End date:
count                             1869
mean     2019-04-08 09:03:10.690208768
min                2019-01-10 00:00:00
25%                2019-01-11 00:00:00
50%                2019-01-11 00:00:00
75%                2019-01-12 00:00:00
max                2020-01-01 00:00:00
Name: end_date, dtype: object


In [25]:
# Fechas de finalización anteriores al inicio
fechas_invalidas = df_contract[
    df_contract['end_date'].notna() &
    (df_contract['end_date'] < df_contract['begin_date'])
]

print(f'Fechas inválidas: {len(fechas_invalidas)}')

Fechas inválidas: 0


In [27]:
# Concatenación de dfs
df= df_contract.copy()

df= df.merge(
    df_personal,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

df= df.merge(
    df_internet,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

df= df.merge(
    df_phone,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

In [37]:
# Revisión de los Nan posterior al merge
print('Clientes sin internet:', df['internet_service'].isna().sum())
print('Clientes sin teléfono:', df['multiple_lines'].isna().sum())

print(
    'Clientes sin internet en df_internet:',
    (~df['customer_id'].isin(df_internet['customer_id'])).sum()
)

print(
    'Clientes sin teléfono en df_phone:',
    (~df['customer_id'].isin(df_phone['customer_id'])).sum()
)

Clientes sin internet: 1526
Clientes sin teléfono: 682
Clientes sin internet en df_internet: 1526
Clientes sin teléfono en df_phone: 682


Se reemplazarán los nan de las columnas 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv' y 'streaming_movies' con 0

Para internet_service se reemplazará con 'No internet'

In [ ]:
# Reemplazo de valores nan
columnas_internet = [
    'online_security',
    'online_backup',
    'device_protection',
    'tech_support',
    'streaming_tv',
    'streaming_movies'
]

df[columnas_internet] = df[columnas_internet].fillna(0)

df['internet_service'] = df['internet_service'].fillna('No internet')

df['multiple_lines'] = df['multiple_lines'].fillna(0)

# Verificación de valores ausentes en todo el dataset
df.isna().sum().sort_values(ascending=False)

## añadir datos (Tenure)
**Importante para añadir tenure sin caer en data leakage**

Si construyesemos tenure asi:
```
df['tenure_months'] = (
    np.where(
        df['end_date'].notna(),
        (df['end_date'] - df['begin_date']).dt.days,
        (fecha_referencia - df['begin_date']).dt.days
    ) / 30.44
)
```
El significado de esta variable sería "Duración total observada de la relación del cliente con la compañía."

Si el objetivo fuera "Clasificar qué clientes terminaron abandonando según las características históricas disponibles en este dataset"

entonces `tenure` calculado con `end_date` puede ser una característica descriptiva.

Pero el objetivo es: **"¿Qué tan probable es que un cliente activo se vaya pronto?"**
Entonces hay un problema. Porque para un cliente que abandonó se estaría calculando su antigüedad hasta después de que ocurrió el evento que se intenta predecir.
```text
Cliente A

2017                 2019
│                     │
Inicio                Abandono
│─────────────────────│
       24 meses
```

El modelo recibe

```
tenure = 24 meses
churn = 1
```

Pero en una predicción real hecha en 2018, todavía no sabríamos que esos 24 meses serían su duración final.

**Diferencia fundamental**

| Variable                   | Qué representa                                | Para "se irá pronto" |
| -------------------------- | --------------------------------------------- | -------------------- |
| `end_date - begin_date`    | Antigüedad **final**                          | ❌ Información futura |
| `fecha_corte - begin_date` | Antigüedad **hasta el momento de predicción** | ✅                    |
| `end_date`                 | Momento en que ocurrió el abandono            | ❌ Leakage directo    |

Por lo que para nuestro objetivo de negocio utilizaremos:

```
fecha_corte = df['begin_date'].max()
```

In [ ]:
# Añade tenure_months y verifica que solo haya valores positivos
fecha_corte = df['begin_date'].max()

df['tenure_months'] = (
    (fecha_corte - df['begin_date']).dt.days / 30.44
)

print(df['tenure_months'].describe())

print(
    'Valores negativos:',
    (df['tenure_months'] < 0).sum()
)

print(
    'Valores cero:',
    (df['tenure_months'] == 0).sum()
)

count    7043.000000
mean       37.296648
std        23.653241
min         0.000000
25%        11.925099
50%        35.742444
75%        59.855453
max        83.705650
Name: tenure_months, dtype: float64
Valores negativos: 0
Valores cero: 11


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        7043 non-null   object        
 1   begin_date         7043 non-null   datetime64[ns]
 2   end_date           1869 non-null   datetime64[ns]
 3   type               7043 non-null   object        
 4   paperless_billing  7043 non-null   int64         
 5   payment_method     7043 non-null   object        
 6   monthly_charges    7043 non-null   float64       
 7   total_charges      7043 non-null   float64       
 8   churn              7043 non-null   int64         
 9   gender             7043 non-null   object        
 10  senior_citizen     7043 non-null   int64         
 11  partner            7043 non-null   int64         
 12  dependents         7043 non-null   int64         
 13  internet_service   7043 non-null   object        
 14  online_s

## Procesamiento de datos previo al modelado y a la división